# Comparison: Base GPT-2 vs DAPT Checkpoint

Loads a base GPT-2 model from OpenAI `.pkl` params, loads a DAPT model from a checkpoint created by `save_checkpoint()`, and compares token embedding vectors using cosine similarity; then looks at changes in next-token probabilities for representative texts.


In [1]:
import os
import sys
import pickle
from pathlib import Path

import torch
import tiktoken

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "notebooks").exists():
            return p
    return start

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)


PROJECT_ROOT: /home/markb/llm-from-scratch
SRC_DIR: /home/markb/llm-from-scratch/src
DATA_DIR: /home/markb/llm-from-scratch/data


In [2]:
from llm_from_scratch.models import gpt2
from llm_from_scratch.training import training_utils
from llm_from_scratch.configs import gpt2small_config
from llm_from_scratch.utils import token_analysis as ta

tokenizer = tiktoken.get_encoding("gpt2")
ta.tokenizer = tokenizer


In [3]:
# Device selection: prefer CUDA if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Default file paths (edit if needed)
BASE_PARAMS_PATH = DATA_DIR / "gpt2_openai_params_124M.pkl"
DAPT_CKPT_PATH = DATA_DIR / "TEST_abstracts_epoch_lastsave_step_lastsave.pth"

print("BASE_PARAMS_PATH exists:", BASE_PARAMS_PATH.exists(), BASE_PARAMS_PATH)
print("DAPT_CKPT_PATH exists:", DAPT_CKPT_PATH.exists(), DAPT_CKPT_PATH)

if not BASE_PARAMS_PATH.exists():
    raise FileNotFoundError(f"Base params file not found: {BASE_PARAMS_PATH}")
if not DAPT_CKPT_PATH.exists():
    raise FileNotFoundError(f"DAPT checkpoint file not found: {DAPT_CKPT_PATH}")


Using device: cpu
BASE_PARAMS_PATH exists: True /home/markb/llm-from-scratch/data/gpt2_openai_params_124M.pkl
DAPT_CKPT_PATH exists: True /home/markb/llm-from-scratch/data/TEST_abstracts_epoch_lastsave_step_lastsave.pth


In [4]:
# 1) Load base model from OpenAI .pkl params
base_cfg = dict(gpt2small_config.GPT_CONFIG_124M_OPENAI)
base_model = gpt2.setup_model(base_cfg).to(device)

with open(BASE_PARAMS_PATH, "rb") as f:
    base_params = pickle.load(f)

training_utils.load_weights_into_gpt(base_model, base_params)
base_model.eval()
print("Loaded base model from:", BASE_PARAMS_PATH)
print("Base tok_emb shape:", tuple(base_model.tok_emb.weight.shape))


Loaded base model from: /home/markb/llm-from-scratch/data/gpt2_openai_params_124M.pkl
Base tok_emb shape: (50257, 768)


In [5]:
# 2) Load DAPT model from .pth checkpoint produced by save_checkpoint()
checkpoint = torch.load(DAPT_CKPT_PATH, map_location=device)

if "model_state_dict" not in checkpoint:
    raise KeyError("Checkpoint does not contain 'model_state_dict'.")

if "run_config" in checkpoint and "model_config" in checkpoint["run_config"]:
    dapt_model_cfg = checkpoint["run_config"]["model_config"]
else:
    # Fallback if run_config is missing
    dapt_model_cfg = base_cfg

dapt_model = gpt2.setup_model(dapt_model_cfg).to(device)
dapt_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
dapt_model.eval()

print("Loaded DAPT model from:", DAPT_CKPT_PATH)
print("DAPT tok_emb shape:", tuple(dapt_model.tok_emb.weight.shape))


Loaded DAPT model from: /home/markb/llm-from-scratch/data/TEST_abstracts_epoch_lastsave_step_lastsave.pth
DAPT tok_emb shape: (50257, 768)


## First analysis blocks: token-pair embedding checks and prompt continuations


In [ ]:
# Extract token embedding matrices (base vs DAPT)
embed_initial = base_model.tok_emb.weight.detach().float().cpu().clone()
embed_after = dapt_model.tok_emb.weight.detach().float().cpu().clone()

print("shape_before:", tuple(embed_initial.shape))
print("shape_after:", tuple(embed_after.shape))


In [ ]:
# Token-pair cosine similarity checks
import pandas as pd

words_dict = {
    "sp_HER2": tokenizer.encode(" HER2"),
    "HER_sp_2": tokenizer.encode("HER 2"),
    "sp_EGFR": tokenizer.encode(" EGFR"),
    "EG_sp_FR": tokenizer.encode("EG FR"),
    "ERBB2_only_BB2": tokenizer.encode("BB2"),
    "ERBB2_only_spERBB": tokenizer.encode(" ERBB"),
    "sp_kinase": tokenizer.encode(" kinase"),
    "king vs dog": [tokenizer.encode("king")[0], tokenizer.encode("dog")[0]],
}

resdf = pd.DataFrame(columns=[
    "word", "tokenid1", "tokenid2", "cos_sim_openai", "cos_sim_aftertrain", "ratio_afterVSbefore"
])

for word, (tokenid1, tokenid2) in words_dict.items():
    cos_sim_openai = ta.compute_cosine_similarity(tokenid1, tokenid2, embed_initial)
    cos_sim_aftertrain = ta.compute_cosine_similarity(tokenid1, tokenid2, embed_after)
    resdf.loc[len(resdf)] = {
        "word": word,
        "tokenid1": tokenid1,
        "tokenid2": tokenid2,
        "cos_sim_openai": cos_sim_openai,
        "cos_sim_aftertrain": cos_sim_aftertrain,
        "ratio_afterVSbefore": cos_sim_aftertrain / cos_sim_openai,
    }

print(resdf.to_string(index=False))


In [ ]:
# 3) Extract token embedding matrices and compute cosine similarity per token for base vs DAPT
base_tok_emb = base_model.tok_emb.weight.detach().float().cpu()
dapt_tok_emb = dapt_model.tok_emb.weight.detach().float().cpu()

if base_tok_emb.shape != dapt_tok_emb.shape:
    raise ValueError(
        f"Embedding shape mismatch: base={tuple(base_tok_emb.shape)} vs dapt={tuple(dapt_tok_emb.shape)}"
    )

cos_scores = ta.cosine_similarity_per_token(base_tok_emb, dapt_tok_emb)
print("cos_scores shape:", tuple(cos_scores.shape))
print("cosine min/max/mean:", cos_scores.min().item(), cos_scores.max().item(), cos_scores.mean().item())


cos_scores shape: (50257,)
cosine min/max/mean: 0.8841298818588257 0.9994819760322571 0.9706948399543762


In [7]:
# 4) Rank top 40 most changed and least changed tokens
TOP_K = 40

most_changed_df = ta.rank_tokens_by_cosine_similarity(
    cosine_scores=cos_scores,
    tokenizer=tokenizer,
    k=TOP_K,
    mode="most_dissimilar",
)

least_changed_df = ta.rank_tokens_by_cosine_similarity(
    cosine_scores=cos_scores,
    tokenizer=tokenizer,
    k=TOP_K,
    mode="most_similar",
)

print("Top 40 MOST changed tokens (lowest cosine):")
print(most_changed_df.to_string(index=False))

print()
print("Top 40 LEAST changed tokens (highest cosine):")
print(least_changed_df.to_string(index=False))


Top 40 MOST changed tokens (lowest cosine):
 tokenid          token  cosine_similarity
     921         ' You'           0.884130
    4705        ' Matt'           0.893998
    3497         ' Get'           0.899295
    5137     ' putting'           0.899304
    6889        ' Make'           0.899364
    1644      ' police'           0.899783
    6035         ' Dan'           0.900174
    2495      ' pretty'           0.900250
    5180       ' Chris'           0.900631
    1639          'You'           0.900823
    4995        ' Mike'           0.901138
    4422        ' Alex'           0.901382
   25508         ' 330'           0.901733
    1526         ' Mar'           0.902345
    5395         ' Jim'           0.902827
    2396           'So'           0.903508
    3932         ' Ben'           0.903555
    6209   ' basically'           0.903645
    7214        ' Take'           0.903873
    3807       ' movie'           0.904015
    1223   ' something'           0.904180
    1532  

In [25]:
# 5) Next-token top-k probabilities for a given prompt
# PROMPT = "Gene amplifications and mutations are common in cancer. A gene that is frequently mutated or amplified is ER"
# PROMPT = "Gene amplifications and mutations are common in cancer. A subtype of breast cancer is HER2-positive breast cancer, which has amplification of ER"
# PROMPT = "Gene amplifications and mutations are common in cancer. A subtype of breast cancer is ER-positive breast cancer, which is different from breast cancer that has amplification of HER"
#PROMPT = "Gene amplifications and mutations are common in cancer. Some breast cancers have more protein expression of HER2 and these are referred to as HER2-positive breast cancers. This protein expression increase is usually due to amplification of ER"
#PROMPT = "Gene amplifications and mutations are common in cancer. Some breast cancers have more protein expression of HER2 and these are referred to as HER2-positive breast cancers. If there is just normal levels of HER2, then this is called HER2-"
PROMPT = "Although HER2 amplification is usually considered in the context of breast cancer, it is clear that it can be amplified in other types of cancer also. In colorectal cancer, a subset of patients are found to have amplification of ER"
# PROMPT = "Cancers that have HER2 amplification have several treatment options, which increase by the year. Notably, the following drugs are aimed at HER2:"
#PROMPT = "Many families like keeping animals in their homes - we call these pets. Among the most popular pets are dogs and"
#PROMPT = "Many families like keeping animals in their homes - we call these pets. Among the most popular pets are dogs, cats and even rodents such as ham"
#PROMPT = "Greek mythology continues to play a significant role in Western culture. Stories of Zeus, Aphrodite and Poseidon still feature in the cinema. And on some pillars you will see in all capital letters the name HER"
#PROMPT = "Greek mythology features the foibles and exploits of Greek gods. To emphasize the importance of Hercules, one author put his name in all caps: HER"
#PROMPT = "There are a number of drugs aimed at the HER2 receptor, and a fair number of these have shown reasonable efficacy in treating HER2-positive breast cancer. Strikingly, many of the tyrosine kinase inhibitors that are aimed at HER2 are used in combination with the monoclonal antibody trastuzumab, but not often with other HER2-directed tyrosine"
#PROMPT = "We were studying the Greek gods and Greek mythology. We had been reading classical Greek literature and learning the myths, especially around the Trojan war. We entered the large temple near Athens and saw that one room was marked ZEUS, one was marked POSEIDON, one was marked ARES, one was marked APHRODITE, and one was marked for the wife of Zeus, the queen of the Greek gods, the key player in the Trojan war and the stories of Io and Leto, whose name is HER"
#PROMPT = "Zeus's wife and the queen of the Greek gods is known as HER"
#PROMPT = "The first portrait was marked LINCOLN, the second was marked DAVID, the third was marked ZEUS, and the fourth was marked without a name, but describing a man who is brave and admired -  HER"
#PROMPT = " HER"
#PROMPT = " ER"
TOP_K_NEXT = 50

def top_next_tokens(model, prompt, context_size, top_k=30):
    idx = training_utils.text_to_token_ids(prompt, tokenizer)
    probas = training_utils.generate_next_token_probability(
        model=model,
        idx=idx,
        context_size=context_size,
    ).squeeze(0).detach().cpu()

    k = min(top_k, probas.numel())
    values, token_ids = torch.topk(probas, k=k, largest=True)

    rows = []
    for rank, (tid, p) in enumerate(zip(token_ids.tolist(), values.tolist()), start=1):
        rows.append(
            {
                "rank": rank,
                "tokenid": tid,
                "token": repr(tokenizer.decode([tid])),
                "probability": float(p),
            }
        )
    return rows

base_context_size = int(base_cfg["context_length"])
dapt_context_size = int(dapt_model_cfg["context_length"])

base_rows = top_next_tokens(base_model, PROMPT, base_context_size, top_k=TOP_K_NEXT)
dapt_rows = top_next_tokens(dapt_model, PROMPT, dapt_context_size, top_k=TOP_K_NEXT)

print("PROMPT:", PROMPT)

print()
print("Base model top next tokens:")
for row in base_rows:
    print(f"{row['rank']:>2}. id={row['tokenid']:<6} token={row['token']:<18} p={row['probability']:.6f}")

print()
print("DAPT model top next tokens:")
for row in dapt_rows:
    print(f"{row['rank']:>2}. id={row['tokenid']:<6} token={row['token']:<18} p={row['probability']:.6f}")


PROMPT: Although HER2 amplification is usually considered in the context of breast cancer, it is clear that it can be amplified in other types of cancer also. In colorectal cancer, a subset of patients are found to have amplification of ER

Base model top next tokens:
 1. id=17394  token='α'                p=0.196708
 2. id=42     token='K'                p=0.111886
 3. id=12     token='-'                p=0.082081
 4. id=38     token='G'                p=0.065900
 5. id=26638  token='β'                p=0.051433
 6. id=47     token='P'                p=0.028462
 7. id=16     token='1'                p=0.018937
 8. id=33     token='B'                p=0.014088
 9. id=2969   token='AP'               p=0.011867
10. id=49     token='R'                p=0.011234
11. id=12016  token='Ps'               p=0.010266
12. id=53     token='V'                p=0.010170
13. id=37     token='F'                p=0.010030
14. id=17     token='2'                p=0.009471
15. id=4778   token=' cells'   

In [ ]:
# Optional: save results to CSV files in output/
OUT_DIR = PROJECT_ROOT / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

most_changed_csv = OUT_DIR / "token_embedding_most_changed_top40.csv"
least_changed_csv = OUT_DIR / "token_embedding_least_changed_top40.csv"

most_changed_df.to_csv(most_changed_csv, index=False)
least_changed_df.to_csv(least_changed_csv, index=False)

print("Saved:", most_changed_csv)
print("Saved:", least_changed_csv)
